# WiSig — Module 2 (Final, streamlined)

Final settings:
- **S_id**: `Fusion_lam0.5 = z(EmbMaha) + 0.5 * z(Energy)` (z-score computed on stable set per fold)
- **S_dom**: `V1_mixNLL` (per-device × per-sourceRX mixture negative log-likelihood; higher ⇒ more drift)
- **Gate** (single-threshold baseline):
  - if `S_id < τ_id` ⇒ **Unknown**
  - else if `S_dom > τ_dom` ⇒ **Known-Drift**
  - else ⇒ **Known-Stable**
- **Threshold calibration** (per fold, on stable set A):
  - `τ_id = quantile(S_id_A, 0.05)`  (FRR target 5%)
  - `τ_dom = quantile(S_dom_A, 0.95)` (false-drift target 5%)

Outputs (per split/fold):
- `gate_scatter.png`, `gate_confmat.png`, `thresholds.json`, `metrics_gate.json`
- `summary_mean_std.json` aggregated over splits×folds

Note: you only need to provide `compact_dataset` by calling your existing loader.


In [1]:
import os, json, time
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pickle

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

dataset_name = "ManySig"
dataset_path = "../ManySig.pkl/"

def load_compact_pkl_dataset(dataset_path,dataset_name):
    with open(dataset_path+dataset_name+'.pkl','rb') as f:
        dataset = pickle.load(f)
    return dataset

# from <your_loader_module> import load_compact_pkl_dataset
compact_dataset = load_compact_pkl_dataset(dataset_path, dataset_name)

# ---- subset sizes ----
TX_TOTAL_USE = 6
RX_TOTAL_USE = 12
DAY_TOTAL_USE = 4

KNOWN_TX_NUM = 4
SOURCE_RX_NUM = 3
TX_SPLIT_REPEATS = 15   # increase for stability study

# ---- protocol ----
TRAIN_DATES = ["2021_03_15"]
TEST_DATES_RX = ["2021_03_15"]
TEST_DATES_DAY = ["2021_03_01"]

# ---- signals ----
Z_FROM_EQ = 1
D_FROM = "raw"
MAX_SIG_PER_TRIPLE = None

# ---- training ----
BATCH_SIZE = 128
EPOCHS = 60
LR = 1e-4
WEIGHT_DECAY = 1e-3
PATIENCE = 8
IN_PLANES = 64
DROPOUT = 0.3

# ---- dims ----
D_DIM = 32

# ---- Sid fusion ----
FUSION_LAM = 0.5

# ---- calibration targets ----
FRR_TARGET = 0.05
FALSE_DRIFT_TARGET = 0.05

# ---- outputs ----
ts = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = f"./results_wisig_module2_final_streamlined/run_{ts}"
os.makedirs(RUN_DIR, exist_ok=True)

cfg = dict(
    SEED=SEED, TX_TOTAL_USE=TX_TOTAL_USE, RX_TOTAL_USE=RX_TOTAL_USE, DAY_TOTAL_USE=DAY_TOTAL_USE,
    KNOWN_TX_NUM=KNOWN_TX_NUM, SOURCE_RX_NUM=SOURCE_RX_NUM, TX_SPLIT_REPEATS=TX_SPLIT_REPEATS,
    TRAIN_DATES=TRAIN_DATES, TEST_DATES_RX=TEST_DATES_RX, TEST_DATES_DAY=TEST_DATES_DAY,
    Z_FROM_EQ=Z_FROM_EQ, D_FROM=D_FROM, MAX_SIG_PER_TRIPLE=MAX_SIG_PER_TRIPLE,
    BATCH_SIZE=BATCH_SIZE, EPOCHS=EPOCHS, LR=LR, WEIGHT_DECAY=WEIGHT_DECAY, PATIENCE=PATIENCE,
    IN_PLANES=IN_PLANES, DROPOUT=DROPOUT, D_DIM=D_DIM,
    FUSION_LAM=FUSION_LAM, FRR_TARGET=FRR_TARGET, FALSE_DRIFT_TARGET=FALSE_DRIFT_TARGET,
)
with open(os.path.join(RUN_DIR, "config.json"), "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2)

print("RUN_DIR:", RUN_DIR)

Device: cuda
RUN_DIR: ./results_wisig_module2_final_streamlined/run_20260305_171514


In [2]:
# =========================
# Data utilities
# =========================
def get_idx(lst, val): 
    return lst.index(val)

def subsample(sigs, max_sig):
    if max_sig is None:
        return sigs
    return sigs[:min(len(sigs), max_sig)]

def get_signals(compact_dataset, tx, rx, date, equalized):
    tx_i = get_idx(compact_dataset["tx_list"], tx)
    rx_i = get_idx(compact_dataset["rx_list"], rx)
    date_i = get_idx(compact_dataset["capture_date_list"], date)
    eq_i = get_idx(compact_dataset["equalized_list"], equalized)
    sigs = compact_dataset["data"][tx_i][rx_i][date_i][eq_i]
    return np.array(sigs, dtype=np.float32)

def iq_to_complex(x_256_2):
    return (x_256_2[...,0] + 1j*x_256_2[...,1]).astype(np.complex64)

def domain_feat_256_complex(x256_c, d_dim=32):
    x = x256_c / (np.sqrt(np.mean(np.abs(x256_c)**2)) + 1e-12)
    X = np.fft.fft(x, n=256)
    mag = np.abs(X) + 1e-12
    logm = np.log(mag)
    D = np.fft.rfft(logm, n=512)
    return np.real(D[:d_dim]).astype(np.float32)

def compute_domain_feats_batch(X_256_2, d_dim=32):
    Xc = iq_to_complex(X_256_2)
    return np.stack([domain_feat_256_complex(Xc[i], d_dim=d_dim) for i in range(Xc.shape[0])], axis=0).astype(np.float32)

In [3]:
# =========================
# Choose subset + random SOURCE RXs (fixed by SEED)
# =========================
tx_list = compact_dataset["tx_list"]
rx_list = compact_dataset["rx_list"]
date_list = compact_dataset["capture_date_list"]

TX_USE = tx_list[:TX_TOTAL_USE]
RX_USE = rx_list[:RX_TOTAL_USE]
DATE_USE = date_list[:DAY_TOTAL_USE]

rng_rx = np.random.RandomState(SEED)
SOURCE_RXS = list(rng_rx.choice(RX_USE, size=SOURCE_RX_NUM, replace=False))
SOURCE_RXS.sort()
DRIFT_RX = [r for r in RX_USE if r not in SOURCE_RXS]

print("TX_USE:", TX_USE)
print("RX_USE:", RX_USE)
print("DATE_USE:", DATE_USE)
print("SOURCE_RXS:", SOURCE_RXS)
print("DRIFT_RX:", DRIFT_RX)

TX_USE: ['14-10', '14-7', '20-15', '20-19', '6-15', '8-20']
RX_USE: ['1-1', '1-19', '14-7', '18-2', '19-2', '2-1', '2-19', '20-1', '3-19', '7-14', '7-7', '8-8']
DATE_USE: ['2021_03_01', '2021_03_08', '2021_03_15', '2021_03_23']
SOURCE_RXS: [np.str_('1-1'), np.str_('7-14'), np.str_('7-7')]
DRIFT_RX: ['1-19', '14-7', '18-2', '19-2', '2-1', '2-19', '20-1', '3-19', '8-8']


In [4]:
# =========================
# Model: ResNet18-1D
# =========================
class BasicBlock1D(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1, downsample=None, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv1d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(p=dropout)
        self.conv2 = nn.Conv1d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(planes)
        self.downsample = downsample
    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.dropout(out)
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.relu(out + identity)

class ResNet18_1D(nn.Module):
    def __init__(self, num_classes, in_planes=64, dropout=0.0):
        super().__init__()
        self.in_planes = in_planes
        self.conv1 = nn.Conv1d(2, in_planes, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(in_planes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(64, 2, stride=1, dropout=dropout)
        self.layer2 = self._make_layer(128, 2, stride=2, dropout=dropout)
        self.layer3 = self._make_layer(256, 2, stride=2, dropout=dropout)
        self.layer4 = self._make_layer(512, 2, stride=2, dropout=dropout)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, planes, blocks, stride, dropout):
        downsample = None
        if stride != 1 or self.in_planes != planes:
            downsample = nn.Sequential(
                nn.Conv1d(self.in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(planes)
            )
        layers = [BasicBlock1D(self.in_planes, planes, stride, downsample, dropout)]
        self.in_planes = planes
        for _ in range(1, blocks):
            layers.append(BasicBlock1D(self.in_planes, planes, dropout=dropout))
        return nn.Sequential(*layers)

    def forward(self, x, return_embed=False):
        x = x.permute(0, 2, 1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x).squeeze(-1)
        logits = self.fc(x)
        if return_embed:
            return logits, x
        return logits

In [5]:
# =========================
# Train / inference helpers
# =========================
class ArrayDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): 
        return self.X.shape[0]
    def __getitem__(self, i): 
        return self.X[i], self.y[i]

def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train(train)
    crit = nn.CrossEntropyLoss()
    total_loss, total_correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        if train:
            optimizer.zero_grad()
        logits = model(xb)
        loss = crit(logits, yb)
        if train:
            loss.backward()
            optimizer.step()
        total_loss += float(loss.item()) * yb.size(0)
        total_correct += int((logits.argmax(1) == yb).sum().item())
        total += int(yb.size(0))
    return total_loss/max(1,total), total_correct/max(1,total)

def infer_logits_embed(model, X_np, batch=512):
    model.eval()
    ds = ArrayDataset(X_np, np.zeros((X_np.shape[0],), dtype=np.int64))
    dl = DataLoader(ds, batch_size=batch, shuffle=False)
    logits_all, Z_all = [], []
    with torch.no_grad():
        for xb,_ in dl:
            xb = xb.to(device)
            logits, emb = model(xb, return_embed=True)
            logits_all.append(logits.cpu().numpy().astype(np.float32))
            Z_all.append(emb.cpu().numpy().astype(np.float32))
    return np.concatenate(logits_all,0), np.concatenate(Z_all,0)

def roc_auc(y_true, score):
    fpr, tpr, _ = roc_curve(y_true, score)
    return float(auc(fpr, tpr))

def l2norm(x, axis=1, eps=1e-12):
    return x/(np.linalg.norm(x,axis=axis,keepdims=True)+eps)

def prototypes(Z, y, K):
    ZN = l2norm(Z, axis=1)
    P = np.zeros((K, Z.shape[1]), dtype=np.float32)
    for k in range(K):
        P[k] = ZN[y==k].mean(axis=0)
    return l2norm(P, axis=1)

def cosine_to_proto(Z, P):
    return l2norm(Z,axis=1) @ P.T

def sid_EmbMaha(Z, mu_z, var_z):
    dif = Z[:,None,:] - mu_z[None,:,:]
    dist = np.sum((dif*dif)/(var_z[None,:,:] + 1e-6), axis=2)
    return (-np.min(dist, axis=1)).astype(np.float32)

def fit_emb_maha_diag(Z_train, y_train, K):
    mu = np.zeros((K, Z_train.shape[1]), dtype=np.float32)
    var = np.zeros((K, Z_train.shape[1]), dtype=np.float32)
    for k in range(K):
        Zk = Z_train[y_train==k]
        mu[k] = Zk.mean(axis=0)
        var[k] = Zk.var(axis=0) + 1e-3
    return mu, var

def sid_Energy(logits):
    m = logits.max(axis=1, keepdims=True)
    return (m + np.log(np.exp(logits-m).sum(axis=1, keepdims=True)+1e-12)).squeeze(1).astype(np.float32)

def zscore(x, ref):
    mu = float(np.mean(ref))
    std = float(np.std(ref))
    if std < 1e-12:
        std = 1.0
    return ((x - mu) / std).astype(np.float32)

In [6]:
# =========================
# Domain (Sdom): V1_mixNLL
# =========================
def mahalanobis_batch(D, mu, Sinv):
    X = D - mu.reshape(1,-1)
    return np.einsum("nd,dd,nd->n", X, Sinv, X).astype(np.float32)

def fit_gaussian(D, reg=1e-3):
    mu = D.mean(axis=0).astype(np.float32)
    C = np.cov(D.T, bias=False).astype(np.float32)
    C = C + reg*np.eye(C.shape[0], dtype=np.float32)
    Sinv = np.linalg.inv(C).astype(np.float32)
    sign, logdet = np.linalg.slogdet(C)
    if sign <= 0:
        logdet = float(np.log(np.maximum(np.linalg.det(C), 1e-12)))
    return mu, Sinv, float(logdet)

def logpdf_gaussian(D, mu, Sinv, logdet):
    d = D.shape[1]
    maha = mahalanobis_batch(D, mu, Sinv)
    return (-0.5*(maha + logdet + d*np.log(2*np.pi))).astype(np.float32)

def fit_device_rx_models(D_train, y_train, rx_train, K, source_rx_ids, reg=1e-3, min_n=20):
    models = {}
    fallback = {}
    for k in range(K):
        fallback[k] = fit_gaussian(D_train[y_train==k], reg=reg)
        for rxid in source_rx_ids:
            idx = np.where((y_train==k) & (rx_train==rxid))[0]
            if idx.size >= min_n:
                models[(k,rxid)] = fit_gaussian(D_train[idx], reg=reg)
    return models, fallback

def sdom_V1_mixNLL(D_eval, k_hat, models_kr, fallback_k, source_rx_ids):
    N = D_eval.shape[0]
    out = np.zeros((N,), dtype=np.float32)
    R = len(source_rx_ids)
    for i in range(N):
        k = int(k_hat[i])
        d = D_eval[i:i+1]
        logps=[]
        for rxid in source_rx_ids:
            mu,Sinv,logdet = models_kr.get((k,rxid), fallback_k[k])
            logps.append(float(logpdf_gaussian(d, mu, Sinv, logdet)[0]))
        logps = np.array(logps, dtype=np.float32)
        m = logps.max()
        loglik = m + np.log(np.exp(logps-m).sum() + 1e-12) - np.log(R)
        out[i] = float(-loglik)  # higher => more drift
    return out

In [7]:
# =========================
# Build datasets
# =========================
def build_source_train(compact_dataset, KNOWN_TX):
    X_list, y_list, D_list = [], [], []
    rx_id_list = []
    for tx in KNOWN_TX:
        for rx in SOURCE_RXS:
            for day in TRAIN_DATES:
                Xz = subsample(get_signals(compact_dataset, tx, rx, day, Z_FROM_EQ), MAX_SIG_PER_TRIPLE)
                Xd = subsample(get_signals(compact_dataset, tx, rx, day, 0 if D_FROM=="raw" else 1), MAX_SIG_PER_TRIPLE)
                n = min(len(Xz), len(Xd))
                Xz = Xz[:n]; Xd = Xd[:n]
                y = np.full((n,), KNOWN_TX.index(tx), dtype=np.int64)
                Df = compute_domain_feats_batch(Xd, d_dim=D_DIM)
                X_list.append(Xz); y_list.append(y); D_list.append(Df)
                rx_id_list.append(np.full((n,), RX_USE.index(rx), dtype=np.int64))
    X = np.concatenate(X_list,0).astype(np.float32)
    y = np.concatenate(y_list,0).astype(np.int64)
    D = np.concatenate(D_list,0).astype(np.float32)
    rx_id = np.concatenate(rx_id_list,0).astype(np.int64)
    return X,y,D,rx_id

def build_eval_set(compact_dataset, KNOWN_TX, txs, rxs, days):
    X_list, D_list = [], []
    for tx in txs:
        for rx in rxs:
            for day in days:
                Xz = subsample(get_signals(compact_dataset, tx, rx, day, Z_FROM_EQ), MAX_SIG_PER_TRIPLE)
                Xd = subsample(get_signals(compact_dataset, tx, rx, day, 0 if D_FROM=="raw" else 1), MAX_SIG_PER_TRIPLE)
                n = min(len(Xz), len(Xd))
                Xz = Xz[:n]; Xd = Xd[:n]
                Df = compute_domain_feats_batch(Xd, d_dim=D_DIM)
                X_list.append(Xz); D_list.append(Df)
    X = np.concatenate(X_list,0).astype(np.float32)
    D = np.concatenate(D_list,0).astype(np.float32)
    return X, D

In [8]:
# =========================
# Gate + plots
# =========================
def calibrate_thresholds(Sid_stable, Sdom_stable, frr_target=0.05, fdr_target=0.05):
    tau_id = float(np.quantile(Sid_stable, frr_target))
    tau_dom = float(np.quantile(Sdom_stable, 1.0 - fdr_target))
    return tau_id, tau_dom

def gate_predict(Sid, Sdom, tau_id, tau_dom):
    # 0=Stable, 1=Drift, 2=Unknown
    pred = np.zeros_like(Sid, dtype=np.int64)
    pred[Sid < tau_id] = 2
    pred[(Sid >= tau_id) & (Sdom > tau_dom)] = 1
    return pred

def plot_gate_scatter(Sid, Sdom, gt, tau_id, tau_dom, out_png, max_points=20000):
    n = len(Sid)
    if n > max_points:
        idx = np.random.RandomState(SEED).choice(n, size=max_points, replace=False)
    else:
        idx = np.arange(n)
    plt.figure(figsize=(6,5))
    plt.scatter(Sid[idx], Sdom[idx], s=3, c=gt[idx], alpha=0.5)
    plt.axvline(tau_id, linestyle="--")
    plt.axhline(tau_dom, linestyle="--")
    plt.xlabel("S_id (Fusion: higher=known)")
    plt.ylabel("S_dom (V1_mixNLL: higher=drift)")
    plt.title("Single-threshold gate (0=stable,1=drift,2=unknown)")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(out_png, dpi=160)
    plt.close()

def plot_confmat(cm, out_png, title):
    plt.figure(figsize=(5,4))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    plt.xticks([0,1,2], ["Stable","Drift","Unknown"])
    plt.yticks([0,1,2], ["Stable","Drift","Unknown"])
    plt.xlabel("Pred"); plt.ylabel("GT")
    plt.tight_layout()
    plt.savefig(out_png, dpi=160)
    plt.close()

In [9]:
# =========================
# Main loop
# =========================
def run_one_split(split_id, KNOWN_TX, UNKNOWN_TX):
    split_dir = os.path.join(RUN_DIR, f"txsplit_{split_id}")
    os.makedirs(split_dir, exist_ok=True)
    with open(os.path.join(split_dir, "tx_split.json"), "w", encoding="utf-8") as f:
        json.dump({"KNOWN_TX": KNOWN_TX, "UNKNOWN_TX": UNKNOWN_TX, "SOURCE_RXS": SOURCE_RXS, "DRIFT_RX": DRIFT_RX}, f, indent=2)

    # source (known TX, source RX, source day)
    X_src, y_src, D_src, rx_src = build_source_train(compact_dataset, KNOWN_TX)
    K = len(KNOWN_TX)

    # drift sets (known TX)
    X_drRX, D_drRX = build_eval_set(compact_dataset, KNOWN_TX, KNOWN_TX, DRIFT_RX, TEST_DATES_RX)
    X_drDAY, D_drDAY = build_eval_set(compact_dataset, KNOWN_TX, KNOWN_TX, SOURCE_RXS, TEST_DATES_DAY)

    # unknown sets (unknown TX)
    X_u_in, D_u_in = build_eval_set(compact_dataset, KNOWN_TX, UNKNOWN_TX, SOURCE_RXS, TEST_DATES_RX)
    X_u_drRX, D_u_drRX = build_eval_set(compact_dataset, KNOWN_TX, UNKNOWN_TX, DRIFT_RX, TEST_DATES_RX)
    X_u_drDAY, D_u_drDAY = build_eval_set(compact_dataset, KNOWN_TX, UNKNOWN_TX, SOURCE_RXS, TEST_DATES_DAY)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + 1000*split_id)
    rows=[]

    for fold,(idx_tr,idx_te) in enumerate(skf.split(X_src, y_src), start=1):
        fold_dir = os.path.join(split_dir, f"fold_{fold}")
        os.makedirs(fold_dir, exist_ok=True)

        # val split
        rng = np.random.RandomState(SEED + 1000*split_id + fold)
        perm = rng.permutation(idx_tr)
        n_val = max(1, int(0.1*len(perm)))
        idx_val = perm[:n_val]
        idx_train = perm[n_val:]

        train_loader = DataLoader(ArrayDataset(X_src[idx_train], y_src[idx_train]), batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(ArrayDataset(X_src[idx_val],   y_src[idx_val]),   batch_size=BATCH_SIZE, shuffle=False)

        model = ResNet18_1D(num_classes=K, in_planes=IN_PLANES, dropout=DROPOUT).to(device)
        opt = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

        best_val=-1.0; best_state=None; patience=0
        hist={"train_loss":[], "train_acc":[], "val_loss":[], "val_acc":[]}

        for ep in range(1, EPOCHS+1):
            tr_loss, tr_acc = run_epoch(model, train_loader, optimizer=opt)
            va_loss, va_acc = run_epoch(model, val_loader, optimizer=None)
            hist["train_loss"].append(tr_loss); hist["train_acc"].append(tr_acc)
            hist["val_loss"].append(va_loss);   hist["val_acc"].append(va_acc)
            if va_acc > best_val + 1e-4:
                best_val = va_acc
                best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1
                if patience >= PATIENCE:
                    break

        torch.save(best_state, os.path.join(fold_dir, "best_model.pth"))
        with open(os.path.join(fold_dir, "history.json"), "w", encoding="utf-8") as f:
            json.dump(hist, f, indent=2)
        model.load_state_dict(best_state)

        # stable A
        logits_A, Z_A = infer_logits_embed(model, X_src[idx_te], batch=512)
        acc = float((np.argmax(logits_A,1) == y_src[idx_te]).mean())
        D_A = D_src[idx_te]

        # train feats for Sid
        logits_tr, Z_tr = infer_logits_embed(model, X_src[idx_train], batch=512)
        mu_z, var_z = fit_emb_maha_diag(Z_tr, y_src[idx_train], K)

        # cache logits+embeddings for eval sets
        logits_B, Z_B = infer_logits_embed(model, X_drRX, batch=512)
        logits_C, Z_C = infer_logits_embed(model, X_drDAY, batch=512)
        logits_D, Z_D = infer_logits_embed(model, X_u_in, batch=512)
        logits_E, Z_E = infer_logits_embed(model, X_u_drRX, batch=512)
        logits_F, Z_F = infer_logits_embed(model, X_u_drDAY, batch=512)

        # Sid components
        SidA_emb = sid_EmbMaha(Z_A, mu_z, var_z)
        SidB_emb = sid_EmbMaha(Z_B, mu_z, var_z)
        SidC_emb = sid_EmbMaha(Z_C, mu_z, var_z)
        SidD_emb = sid_EmbMaha(Z_D, mu_z, var_z)
        SidE_emb = sid_EmbMaha(Z_E, mu_z, var_z)
        SidF_emb = sid_EmbMaha(Z_F, mu_z, var_z)

        SidA_en = sid_Energy(logits_A)
        SidB_en = sid_Energy(logits_B)
        SidC_en = sid_Energy(logits_C)
        SidD_en = sid_Energy(logits_D)
        SidE_en = sid_Energy(logits_E)
        SidF_en = sid_Energy(logits_F)

        # Fusion Sid: z-score each component on stable A, then sum
        zA_emb = zscore(SidA_emb, SidA_emb); zB_emb = zscore(SidB_emb, SidA_emb); zC_emb = zscore(SidC_emb, SidA_emb)
        zD_emb = zscore(SidD_emb, SidA_emb); zE_emb = zscore(SidE_emb, SidA_emb); zF_emb = zscore(SidF_emb, SidA_emb)

        zA_en  = zscore(SidA_en, SidA_en);  zB_en  = zscore(SidB_en, SidA_en);  zC_en  = zscore(SidC_en, SidA_en)
        zD_en  = zscore(SidD_en, SidA_en);  zE_en  = zscore(SidE_en, SidA_en);  zF_en  = zscore(SidF_en, SidA_en)

        Sid_A = (zA_emb + FUSION_LAM*zA_en).astype(np.float32)
        Sid_B = (zB_emb + FUSION_LAM*zB_en).astype(np.float32)
        Sid_C = (zC_emb + FUSION_LAM*zC_en).astype(np.float32)
        Sid_D = (zD_emb + FUSION_LAM*zD_en).astype(np.float32)
        Sid_E = (zE_emb + FUSION_LAM*zE_en).astype(np.float32)
        Sid_F = (zF_emb + FUSION_LAM*zF_en).astype(np.float32)

        # k_hat for Sdom
        P = prototypes(Z_tr, y_src[idx_train], K)
        kA = np.argmax(cosine_to_proto(Z_A, P),1).astype(np.int64)
        kB = np.argmax(cosine_to_proto(Z_B, P),1).astype(np.int64)
        kC = np.argmax(cosine_to_proto(Z_C, P),1).astype(np.int64)
        kD = np.argmax(cosine_to_proto(Z_D, P),1).astype(np.int64)
        kE = np.argmax(cosine_to_proto(Z_E, P),1).astype(np.int64)
        kF = np.argmax(cosine_to_proto(Z_F, P),1).astype(np.int64)

        # Fit domain models on train split only
        source_rx_ids = sorted({RX_USE.index(r) for r in SOURCE_RXS})
        models_kr, fb = fit_device_rx_models(D_src[idx_train], y_src[idx_train], rx_src[idx_train], K, source_rx_ids, reg=1e-3, min_n=20)

        Sdom_A = sdom_V1_mixNLL(D_A, kA, models_kr, fb, source_rx_ids)
        Sdom_B = sdom_V1_mixNLL(D_drRX, kB, models_kr, fb, source_rx_ids)
        Sdom_C = sdom_V1_mixNLL(D_drDAY, kC, models_kr, fb, source_rx_ids)
        Sdom_D = sdom_V1_mixNLL(D_u_in, kD, models_kr, fb, source_rx_ids)
        Sdom_E = sdom_V1_mixNLL(D_u_drRX, kE, models_kr, fb, source_rx_ids)
        Sdom_F = sdom_V1_mixNLL(D_u_drDAY, kF, models_kr, fb, source_rx_ids)

        # thresholds on stable A
        tau_id, tau_dom = calibrate_thresholds(Sid_A, Sdom_A, FRR_TARGET, FALSE_DRIFT_TARGET)
        with open(os.path.join(fold_dir, "thresholds.json"), "w", encoding="utf-8") as f:
            json.dump({"tau_id": tau_id, "tau_dom": tau_dom}, f, indent=2)

        # pooled set for gating
        Sid_all = np.concatenate([Sid_A, Sid_B, Sid_C, Sid_D, Sid_E, Sid_F])
        Sdom_all = np.concatenate([Sdom_A, Sdom_B, Sdom_C, Sdom_D, Sdom_E, Sdom_F])
        gt = np.concatenate([
            np.zeros_like(Sid_A, dtype=np.int64),
            np.ones_like(Sid_B, dtype=np.int64),
            np.ones_like(Sid_C, dtype=np.int64),
            np.full_like(Sid_D, 2, dtype=np.int64),
            np.full_like(Sid_E, 2, dtype=np.int64),
            np.full_like(Sid_F, 2, dtype=np.int64),
        ])

        pred = gate_predict(Sid_all, Sdom_all, tau_id, tau_dom)
        cm = confusion_matrix(gt, pred, labels=[0,1,2])

        plot_gate_scatter(Sid_all, Sdom_all, gt, tau_id, tau_dom, os.path.join(fold_dir, "gate_scatter.png"))
        plot_confmat(cm, os.path.join(fold_dir, "gate_confmat.png"), "3-class gating confusion")

        # operating metrics
        FRR_stable = float(1.0 - (pred[gt==0] == 0).mean())
        false_drift_stable = float((pred[gt==0] == 1).mean())
        FAR_unknown_accept = float((pred[gt==2] == 0).mean())
        miss_drift = float((pred[gt==1] == 0).mean())

        # AUCs
        auc_unknown = roc_auc(
            np.concatenate([np.zeros_like(Sid_A, dtype=np.int64), np.ones_like(Sid_D, dtype=np.int64)]),
            np.concatenate([-Sid_A, -Sid_D])
        )
        auc_drift_rx = roc_auc(
            np.concatenate([np.zeros_like(Sdom_A, dtype=np.int64), np.ones_like(Sdom_B, dtype=np.int64)]),
            np.concatenate([Sdom_A, Sdom_B])
        )
        auc_drift_day = roc_auc(
            np.concatenate([np.zeros_like(Sdom_A, dtype=np.int64), np.ones_like(Sdom_C, dtype=np.int64)]),
            np.concatenate([Sdom_A, Sdom_C])
        )

        metrics = dict(
            split=split_id, fold=fold, acc=acc,
            auc_unknown=auc_unknown, auc_drift_rx=auc_drift_rx, auc_drift_day=auc_drift_day,
            tau_id=tau_id, tau_dom=tau_dom,
            FRR_stable=FRR_stable, false_drift_stable=false_drift_stable,
            FAR_unknown_accept=FAR_unknown_accept, miss_drift=miss_drift,
            confusion_matrix=cm.tolist(),
        )
        with open(os.path.join(fold_dir, "metrics_gate.json"), "w", encoding="utf-8") as f:
            json.dump(metrics, f, indent=2)

        rows.append(metrics)
        print(f"[split {split_id} fold {fold}] acc={acc:.3f} AUC_u={auc_unknown:.3f} AUC_drRX={auc_drift_rx:.3f} FARunk={FAR_unknown_accept:.3f} missDr={miss_drift:.3f}")

    with open(os.path.join(split_dir, "metrics_all_folds.json"), "w", encoding="utf-8") as f:
        json.dump(rows, f, indent=2)
    return rows

all_rows=[]
for split_id in range(1, TX_SPLIT_REPEATS+1):
    rng_tx = np.random.RandomState(SEED + 777*split_id)
    tx_perm = list(rng_tx.permutation(TX_USE))
    KNOWN_TX = tx_perm[:KNOWN_TX_NUM]
    UNKNOWN_TX = tx_perm[KNOWN_TX_NUM:]
    print("\n=== TX split", split_id, "===")
    print("KNOWN_TX:", KNOWN_TX)
    print("UNKNOWN_TX:", UNKNOWN_TX)
    all_rows.extend(run_one_split(split_id, KNOWN_TX, UNKNOWN_TX))

with open(os.path.join(RUN_DIR, "metrics_all_splits_folds.json"), "w", encoding="utf-8") as f:
    json.dump(all_rows, f, indent=2)

print("\nSaved all metrics to:", RUN_DIR)


=== TX split 1 ===
KNOWN_TX: [np.str_('20-15'), np.str_('20-19'), np.str_('14-10'), np.str_('8-20')]
UNKNOWN_TX: [np.str_('6-15'), np.str_('14-7')]
[split 1 fold 1] acc=0.999 AUC_u=0.812 AUC_drRX=0.941 FARunk=0.393 missDr=0.139
[split 1 fold 2] acc=0.997 AUC_u=0.825 AUC_drRX=0.946 FARunk=0.392 missDr=0.149
[split 1 fold 3] acc=0.998 AUC_u=0.821 AUC_drRX=0.953 FARunk=0.131 missDr=0.141
[split 1 fold 4] acc=0.998 AUC_u=0.772 AUC_drRX=0.945 FARunk=0.433 missDr=0.149
[split 1 fold 5] acc=0.997 AUC_u=0.664 AUC_drRX=0.947 FARunk=0.370 missDr=0.154

=== TX split 2 ===
KNOWN_TX: [np.str_('8-20'), np.str_('6-15'), np.str_('20-15'), np.str_('14-10')]
UNKNOWN_TX: [np.str_('20-19'), np.str_('14-7')]
[split 2 fold 1] acc=0.998 AUC_u=0.993 AUC_drRX=0.923 FARunk=0.000 missDr=0.247
[split 2 fold 2] acc=0.998 AUC_u=0.996 AUC_drRX=0.922 FARunk=0.000 missDr=0.240
[split 2 fold 3] acc=0.998 AUC_u=0.992 AUC_drRX=0.919 FARunk=0.000 missDr=0.215
[split 2 fold 4] acc=0.999 AUC_u=0.985 AUC_drRX=0.927 FARunk=0

In [10]:
# =========================
# Summary
# =========================
def mean_std(vals):
    vals = np.array(vals, dtype=np.float64)
    return float(vals.mean()), float(vals.std(ddof=0))

summary = {}
for key in ["acc","auc_unknown","auc_drift_rx","auc_drift_day","FRR_stable","false_drift_stable","FAR_unknown_accept","miss_drift","tau_id","tau_dom"]:
    m,s = mean_std([r[key] for r in all_rows])
    summary[key] = {"mean": m, "std": s}

cm_sum = np.zeros((3,3), dtype=np.int64)
for r in all_rows:
    cm_sum += np.array(r["confusion_matrix"], dtype=np.int64)
summary["confusion_matrix_sum"] = cm_sum.tolist()

with open(os.path.join(RUN_DIR, "summary_mean_std.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("=== Mean ± Std over splits×folds ===")
for k,v in summary.items():
    if k == "confusion_matrix_sum":
        continue
    print(f"{k:18s}: {v['mean']:.3f} ± {v['std']:.3f}")

print("\nConfusion matrix SUM (rows=GT, cols=Pred) [Stable,Drift,Unknown]:")
print(cm_sum)

=== Mean ± Std over splits×folds ===
acc               : 0.997 ± 0.002
auc_unknown       : 0.936 ± 0.103
auc_drift_rx      : 0.934 ± 0.010
auc_drift_day     : 0.842 ± 0.021
FRR_stable        : 0.093 ± 0.003
false_drift_stable: 0.043 ± 0.003
FAR_unknown_accept: 0.053 ± 0.111
miss_drift        : 0.235 ± 0.048
tau_id            : -1.279 ± 0.193
tau_dom           : 2335.432 ± 285.183

Confusion matrix SUM (rows=GT, cols=Pred) [Stable,Drift,Unknown]:
[[ 163204    7796    9000]
 [ 844986  571601 2183413]
 [ 119632  285983 1844385]]
